In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

drive_root = Path("/content/drive/MyDrive/vietnamese-summarization")

if not drive_root.is_dir():
    raise RuntimeError("Lỗi: Không tìm thấy thư mục dự án trên Drive.")

print("Đã kết nối Drive.")

In [ ]:
import os
import sys
import subprocess

repo_url = "https://github.com/MD1809/vietnamese-summarization.git"
repo_dir = Path("/content/vietnamese-summarization")
target_commit = "bd92efd65cd21ed2add88defd3a110e931f5d573"

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", repo_url, str(repo_dir)],
        check=True,
    )

if not (repo_dir / ".git").is_dir():
    raise RuntimeError("Lỗi: Thư mục dự án không phải Git repository.")

remote_result = subprocess.run(
    ["git", "remote", "get-url", "origin"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

if remote_result.stdout.strip() != repo_url:
    raise RuntimeError("Lỗi: Repository có origin khác với dự án.")

status_result = subprocess.run(
    ["git", "status", "--porcelain"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

if status_result.stdout.strip() != "":
    raise RuntimeError(
        "Lỗi: Repository có thay đổi chưa lưu. "
        "Cần kiểm tra trước khi chuyển commit."
    )

subprocess.run(
    ["git", "fetch", "origin"],
    cwd=repo_dir,
    check=True,
)

subprocess.run(
    ["git", "checkout", "--detach", target_commit],
    cwd=repo_dir,
    check=True,
)

os.chdir(repo_dir)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print("Commit đang sử dụng:", target_commit)
print("Thư mục làm việc:", Path.cwd())

In [ ]:
%pip uninstall -y diffusers
%pip install "jedi==0.20.1"
%pip install -r /content/vietnamese-summarization/requirements.txt
%pip check

In [ ]:
from google.colab import files

uploaded = files.upload()

if len(uploaded) != 1:
    raise RuntimeError("Lỗi: Hãy chọn đúng một file data.py.")

for file_name in uploaded:
    uploaded_content = uploaded[file_name]

data_module_path = repo_dir / "src" / "data.py"
data_module_path.write_bytes(uploaded_content)

print("File đã chọn:", file_name)
print("Đã cập nhật:", data_module_path)

In [ ]:
import json
from src.data import load_wikilingua, save_source_manifest

cache_dir = drive_root / "data" / "raw"
manifest_path = drive_root / "data" / "manifests" / "wikilingua_source.json"

dataset = load_wikilingua(cache_dir)

manifest = save_source_manifest(
    dataset,
    manifest_path,
    cache_dir,
)

print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("Đã lưu manifest:", manifest_path)

In [ ]:
files.download(str(manifest_path))

In [ ]:
import importlib
import src.data

importlib.reload(src.data)

source_rows = dataset["train"]

if len(source_rows) < 10:
    raise RuntimeError("Lỗi: Không đủ 10 bài để khảo sát.")

for row_index in range(10):
    row = source_rows[row_index]

    print("=" * 80)
    print("Vị trí bài:", row_index)
    print("URL:", row["url"])
    print("Kiểu dữ liệu article:", type(row["article"]).__name__)
    print(json.dumps(row["article"], ensure_ascii=False, indent=2))

In [ ]:
pairs = src.data.convert_wikilingua_sections(source_rows)

expected_pair_count = 0

for row in source_rows:
    section_count = len(row["article"]["document"])
    expected_pair_count = expected_pair_count + section_count

if len(pairs) != expected_pair_count:
    raise RuntimeError("Lỗi: Số cặp không khớp tổng số section nguồn.")

print(json.dumps(
    {
        "article_count": len(source_rows),
        "section_count": expected_pair_count,
        "pair_count": len(pairs),
        "pair_count_check_passed": True,
    },
    ensure_ascii=False,
    indent=2,
))

In [ ]:
selected_pairs = []
selected_rows = []

for pair in pairs:
    row_index = pair["source_row_index"]

    if row_index not in selected_rows:
        selected_pairs.append(pair)
        selected_rows.append(row_index)

    if len(selected_pairs) == 5:
        break

if len(selected_pairs) != 5:
    raise RuntimeError("Lỗi: Không đủ mẫu từ 5 bài khác nhau.")

sample_number = 0

for pair in selected_pairs:
    sample_number = sample_number + 1
    row_index = pair["source_row_index"]
    section_index = pair["source_section_index"]

    source_row = source_rows[row_index]
    source_article = source_row["article"]

    if pair["url"] != source_row["url"]:
        raise RuntimeError("Lỗi: URL không khớp bài nguồn.")

    for field in ["section_name", "document", "summary"]:
        if pair[field] != source_article[field][section_index]:
            raise RuntimeError(
                f"Lỗi: Trường {field} không khớp section nguồn."
            )

    print("=" * 80)
    print("Mẫu đối chiếu:", sample_number)
    print(json.dumps(pair, ensure_ascii=False, indent=2))
    print("Đối chiếu giá trị với section nguồn: đạt.")

In [ ]:
import importlib
from google.colab import files

uploaded = files.upload()

if len(uploaded) != 1:
    raise RuntimeError("Lỗi: Hãy chọn đúng một file preprocessing.py.")

for file_name in uploaded:
    uploaded_content = uploaded[file_name]

module_path = repo_dir / "src" / "preprocessing.py"
module_path.write_bytes(uploaded_content)

import src.preprocessing

importlib.reload(src.preprocessing)

print("File đã chọn:", file_name)
print("Đã cập nhật:", module_path)

In [ ]:
import json
import unicodedata

if "pairs" not in globals():
    raise RuntimeError("Lỗi: Chưa có dữ liệu cặp từ cell 8.")

normalized_pairs = src.preprocessing.normalize_pairs(pairs)

if len(normalized_pairs) != len(pairs):
    raise RuntimeError("Lỗi: Số cặp thay đổi sau chuẩn hóa.")

changed_counts = {
    "document": 0,
    "summary": 0,
}

for pair_index in range(len(pairs)):
    original_pair = pairs[pair_index]
    normalized_pair = normalized_pairs[pair_index]

    for field in ["document", "summary"]:
        original_value = original_pair.get(field)
        normalized_value = normalized_pair.get(field)

        if isinstance(original_value, str):
            if original_value != normalized_value:
                changed_counts[field] = changed_counts[field] + 1

            if not unicodedata.is_normalized("NFC", normalized_value):
                raise RuntimeError("Lỗi: Văn bản chưa ở dạng NFC.")

            checked_value = src.preprocessing.normalize_text(
                normalized_value
            )

            if checked_value != normalized_value:
                raise RuntimeError("Lỗi: Chuẩn hóa chưa ổn định.")

        else:
            if normalized_value != original_value:
                raise RuntimeError("Lỗi: Giá trị không phải chuỗi bị thay đổi.")

print(json.dumps(
    {
        "pair_count_before": len(pairs),
        "pair_count_after": len(normalized_pairs),
        "changed_document_count": changed_counts["document"],
        "changed_summary_count": changed_counts["summary"],
        "pair_count_check_passed": True,
        "nfc_check_passed": True,
        "idempotence_check_passed": True,
    },
    ensure_ascii=False,
    indent=2,
))

In [ ]:
displayed_count = 0

for pair_index in range(len(pairs)):
    original_pair = pairs[pair_index]
    normalized_pair = normalized_pairs[pair_index]

    for field in ["document", "summary"]:
        original_value = original_pair.get(field)
        normalized_value = normalized_pair.get(field)

        if isinstance(original_value, str):
            if original_value != normalized_value:
                displayed_count = displayed_count + 1

                print("=" * 80)
                print("Mẫu đối chiếu:", displayed_count)
                print("Vị trí cặp:", pair_index)
                print("URL:", original_pair["url"])
                print("Section:", original_pair["section_name"])
                print("Trường:", field)
                print("Trước:", repr(original_value))
                print("Sau:", repr(normalized_value))

        if displayed_count == 3:
            break

    if displayed_count == 3:
        break

if displayed_count == 0:
    print("Không có trường dạng chuỗi nào thay đổi sau chuẩn hóa.")

In [ ]:
import importlib
from google.colab import files

uploaded = files.upload()

if len(uploaded) != 1:
    raise RuntimeError("Lỗi: Hãy chọn đúng một file data.py.")

for file_name in uploaded:
    uploaded_content = uploaded[file_name]

module_path = repo_dir / "src" / "data.py"
module_path.write_bytes(uploaded_content)

import src.data

importlib.reload(src.data)

print("File đã chọn:", file_name)
print("Đã cập nhật:", module_path)

In [ ]:
import json

if "normalized_pairs" not in globals():
    raise RuntimeError("Lỗi: Chưa có dữ liệu chuẩn hóa từ cell 11.")

cleaned_pairs, cleaning_report = src.data.clean_wikilingua_pairs(
    normalized_pairs
)

retained_ids = set()
retained_contents = set()

for pair in cleaned_pairs:
    sample_id = pair["id"]
    content = (pair["document"], pair["summary"])

    if sample_id in retained_ids:
        raise RuntimeError("Lỗi: ID bị trùng trong dữ liệu giữ lại.")

    if content in retained_contents:
        raise RuntimeError("Lỗi: Vẫn còn cặp document-summary trùng.")

    if pair["group_id"] != pair["url"]:
        raise RuntimeError("Lỗi: group_id không khớp URL bài gốc.")

    retained_ids.add(sample_id)
    retained_contents.add(content)

removed_count = 0

for reason in cleaning_report["removed_counts_by_reason"]:
    removed_count = (
        removed_count
        + cleaning_report["removed_counts_by_reason"][reason]
    )

if removed_count != cleaning_report["removed_pair_count"]:
    raise RuntimeError("Lỗi: Tổng số loại theo lý do không khớp.")

if len(cleaned_pairs) + removed_count != len(normalized_pairs):
    raise RuntimeError("Lỗi: Tổng số giữ lại và bị loại không khớp đầu vào.")

print(json.dumps(
    {
        "input_pair_count": cleaning_report["input_pair_count"],
        "retained_pair_count": cleaning_report["retained_pair_count"],
        "removed_pair_count": cleaning_report["removed_pair_count"],
        "removed_counts_by_reason": cleaning_report["removed_counts_by_reason"],
        "shared_source_group_count": len(
            cleaning_report["shared_source_groups"]
        ),
        "unique_id_check_passed": True,
        "exact_duplicate_check_passed": True,
        "group_id_check_passed": True,
        "count_check_passed": True,
    },
    ensure_ascii=False,
    indent=2,
))

In [ ]:
cleaning_manifest_path = (
    drive_root
    / "data"
    / "manifests"
    / "wikilingua_cleaning.json"
)

cleaning_manifest = src.data.save_cleaning_manifest(
    cleaning_report,
    cleaning_manifest_path,
)

print("Đã lưu manifest:", cleaning_manifest_path)

files.download(str(cleaning_manifest_path))

In [ ]:
import importlib
from google.colab import files

uploaded = files.upload()

if len(uploaded) != 1:
    raise RuntimeError("Lỗi: Hãy chọn đúng một file data.py.")

for file_name in uploaded:
    uploaded_content = uploaded[file_name]

module_path = repo_dir / "src" / "data.py"
module_path.write_bytes(uploaded_content)

import src.data

importlib.reload(src.data)

print("File đã chọn:", file_name)
print("Đã cập nhật:", module_path)

In [ ]:
import json

if "cleaned_pairs" not in globals():
    raise RuntimeError("Lỗi: Chưa có dữ liệu sạch từ cell 14.")

if "cleaning_report" not in globals():
    raise RuntimeError("Lỗi: Chưa có báo cáo làm sạch từ cell 14.")

dataset_splits, split_manifest = src.data.split_wikilingua_pairs(
    cleaned_pairs,
    cleaning_report,
)

repeated_splits, repeated_manifest = src.data.split_wikilingua_pairs(
    cleaned_pairs,
    cleaning_report,
)

if repeated_manifest != split_manifest:
    raise RuntimeError("Lỗi: Bộ chia thay đổi khi chạy lại cùng dữ liệu và seed.")

split_summary = {}

for split_name in dataset_splits:
    metadata = split_manifest["splits"][split_name]

    split_summary[split_name] = {
        "pair_count": metadata["pair_count"],
        "article_group_count": metadata["article_group_count"],
        "split_group_count": metadata["split_group_count"],
    }

print(json.dumps(
    {
        "input_pair_count": split_manifest["input_pair_count"],
        "seed": split_manifest["seed"],
        "splits": split_summary,
        "checks": split_manifest["checks"],
        "reproducibility_check_passed": True,
    },
    ensure_ascii=False,
    indent=2,
))

del repeated_splits
del repeated_manifest

In [ ]:
split_manifest_path = (
    drive_root
    / "data"
    / "manifests"
    / "wikilingua_splits.json"
)

saved_split_manifest = src.data.save_split_manifest(
    split_manifest,
    split_manifest_path,
)

print("Đã lưu manifest:", split_manifest_path)

files.download(str(split_manifest_path))

In [ ]:
import importlib
from google.colab import files

uploaded = files.upload()

if len(uploaded) != 1:
    raise RuntimeError("Lỗi: Hãy chọn đúng một file data.py.")

for file_name in uploaded:
    uploaded_content = uploaded[file_name]

module_path = repo_dir / "src" / "data.py"
module_path.write_bytes(uploaded_content)

import src.data

importlib.reload(src.data)

print("File đã chọn:", file_name)
print("Đã cập nhật:", module_path)

In [ ]:
import json

if "dataset_splits" not in globals():
    raise RuntimeError("Lỗi: Chưa có bộ chia từ cell 17.")

if "split_manifest" not in globals():
    raise RuntimeError("Lỗi: Chưa có manifest bộ chia từ cell 17.")

target_counts = {
    "train": 2000,
    "validation": 200,
    "test": 200,
}

subset_manifest_path = (
    drive_root
    / "data"
    / "manifests"
    / "wikilingua_subsets.json"
)

if subset_manifest_path.is_file():
    subset_manifest = json.loads(
        subset_manifest_path.read_text(encoding="utf-8")
    )

    for split_name in target_counts:
        saved_target = subset_manifest["splits"][split_name]["target_pair_count"]

        if saved_target != target_counts[split_name]:
            raise RuntimeError("Lỗi: Mục tiêu trong manifest khác mốc đã chốt.")

    subset_splits = src.data.restore_fixed_subsets(
        dataset_splits,
        split_manifest,
        subset_manifest,
    )

    print("Đã khôi phục tập con theo ID trong manifest.")

else:
    subset_splits, subset_manifest = src.data.select_fixed_subsets(
        dataset_splits,
        split_manifest,
        target_counts,
    )

    print("Đã chọn tập con lần đầu.")

restored_subsets = src.data.restore_fixed_subsets(
    dataset_splits,
    split_manifest,
    subset_manifest,
)

if restored_subsets != subset_splits:
    raise RuntimeError("Lỗi: Khôi phục theo ID không khớp tập con đã chọn.")

subset_summary = {}

for split_name in subset_splits:
    metadata = subset_manifest["splits"][split_name]

    subset_summary[split_name] = {
        "target_pair_count": metadata["target_pair_count"],
        "actual_pair_count": metadata["pair_count"],
        "parent_pair_count": metadata["parent_pair_count"],
        "article_group_count": metadata["article_group_count"],
        "split_group_count": metadata["split_group_count"],
    }

print(json.dumps(
    {
        "seed": subset_manifest["seed"],
        "splits": subset_summary,
        "checks": subset_manifest["checks"],
        "saved_id_restore_check_passed": True,
    },
    ensure_ascii=False,
    indent=2,
))

del restored_subsets

In [ ]:
saved_subset_manifest = src.data.save_subset_manifest(
    subset_manifest,
    subset_manifest_path,
)

print("Đã lưu manifest:", subset_manifest_path)

files.download(str(subset_manifest_path))

In [ ]:
import importlib
from google.colab import files

uploaded = files.upload()

if len(uploaded) != 1:
    raise RuntimeError("Lỗi: Hãy chọn đúng một file data.py.")

for file_name in uploaded:
    uploaded_content = uploaded[file_name]

module_path = repo_dir / "src" / "data.py"
module_path.write_bytes(uploaded_content)

import src.data

importlib.reload(src.data)

print("File đã chọn:", file_name)
print("Đã cập nhật:", module_path)

In [ ]:
import json

for variable_name in [
    "dataset_splits",
    "split_manifest",
    "cleaning_report",
]:
    if variable_name not in globals():
        raise RuntimeError(f"Lỗi: Chưa có biến {variable_name}.")

subset_manifest_path = (
    drive_root
    / "data"
    / "manifests"
    / "wikilingua_subsets.json"
)

subset_manifest = json.loads(
    subset_manifest_path.read_text(encoding="utf-8")
)

subset_splits = src.data.restore_fixed_subsets(
    dataset_splits,
    split_manifest,
    subset_manifest,
)

dataset_statistics = src.data.build_dataset_statistics(
    subset_splits,
    subset_manifest,
    cleaning_report,
)

print(json.dumps(
    dataset_statistics,
    ensure_ascii=False,
    indent=2,
))

In [ ]:
processed_manifest = src.data.export_prepared_datasets(
    subset_splits,
    subset_manifest,
    dataset_statistics,
    drive_root,
)

print(json.dumps(
    processed_manifest,
    ensure_ascii=False,
    indent=2,
))

print("Đã kiểm tra đọc lại JSONL và đối chiếu tập con cố định.")
print("Đã ghi nhận khóa tập test.")

In [ ]:
for manifest_name in [
    "wikilingua_statistics.json",
    "wikilingua_processed.json",
]:
    manifest_path = drive_root / "data" / "manifests" / manifest_name
    files.download(str(manifest_path))

In [ ]:
from google.colab import files
from src.utils import load_train_config

uploaded = files.upload()

if len(uploaded) != 1:
    raise RuntimeError("Lỗi: Hãy chọn đúng một file train.json.")

for file_name in uploaded:
    uploaded_content = uploaded[file_name]

config_path = repo_dir / "configs" / "train.json"
config_path.write_bytes(uploaded_content)

train_config = load_train_config()

print("Đã cập nhật:", config_path)
print("Giới hạn nguồn:", train_config["max_source_length"])
print("Giới hạn đích:", train_config["max_target_length"])
print("Tiền tố tác vụ:", repr(train_config["task_prefix"]))

In [ ]:
import importlib

uploaded = files.upload()

if len(uploaded) != 1:
    raise RuntimeError("Lỗi: Hãy chọn đúng một file tokenization.py.")

for file_name in uploaded:
    uploaded_content = uploaded[file_name]

module_path = repo_dir / "src" / "tokenization.py"
module_path.write_bytes(uploaded_content)

import src.tokenization

importlib.reload(src.tokenization)

print("Đã cập nhật:", module_path)

In [ ]:
import json
from src.utils import load_train_config

train_config = load_train_config()

if train_config["max_source_length"] != 1024:
    raise RuntimeError("Lỗi: Giới hạn nguồn phải là 1024.")

if train_config["max_target_length"] != 128:
    raise RuntimeError("Lỗi: Giới hạn đích phải là 128.")

truncation_report = src.tokenization.measure_truncation(
    drive_root,
)

print(json.dumps(
    truncation_report,
    ensure_ascii=False,
    indent=2,
))

In [ ]:
truncation_report_path = (
    drive_root
    / "data"
    / "manifests"
    / "wikilingua_truncation_1024_128.json"
)

saved_truncation_report = src.tokenization.save_truncation_report(
    truncation_report,
    truncation_report_path,
)

print("Đã lưu báo cáo:", truncation_report_path)

files.download(str(truncation_report_path))